# Demonstrating `build_wcs_from_xradio`

This notebook demonstrates how to use the `build_wcs_from_xradio` function from the `wcs_reproject` module to create an `astropy.wcs.WCS` object from an `xarray.DataArray` with XRADIO-style metadata.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import xarray as xr
from wcs_reproject import build_wcs_from_xradio
from xradio.image import make_empty_sky_image

# 1. Create a synthetic XRADIO image using make_empty_sky_image
phase_center = [0.0, 0.0]  # [ra, dec] in rad
image_size = [256, 256]
cell_arcsec = 2.0
cell = np.deg2rad(cell_arcsec / 3600.0)
cell_size = [cell, cell]
frequency_coords = np.array([1.4e9])
pol_coords = ['I']
time_coords = np.array([59000.0])

xds = make_empty_sky_image(
    phase_center=phase_center,
    image_size=image_size,
    cell_size=cell_size,
    frequency_coords=frequency_coords,
    pol_coords=pol_coords,
    time_coords=time_coords,
    direction_reference='fk5',
    projection='SIN',
    spectral_reference='lsrk',
    do_sky_coords=True,
)

# The build_wcs_from_xradio function expects a DataArray with coordinate_system_info in its attrs.
# We can create a dummy SKY DataArray and attach the attrs from the Dataset.
dims = ('time', 'frequency', 'polarization', 'l', 'm')
coords = {d: xds.coords[d] for d in dims}
sky_shape = tuple(len(coords[d]) for d in dims)
sky_data = np.zeros(sky_shape, dtype=np.float32)
sky = xr.DataArray(sky_data, dims=dims, coords=coords)
sky["units"] = "Jy/pixel"
xds["SKY"] = sky
sky.attrs.update(xds.attrs)

# 2. Call build_wcs_from_xradio
wcs_build_result = build_wcs_from_xradio(sky, dim_a='l', dim_b='m')
wcs = wcs_build_result.wcs

# 3. Print the resulting WCS object
print(wcs)